# Lecture 4b — Serving with MLflow: Models, Registry, `mlflow models serve`
**AI Operations · Module 4 · CPU only · core path ≈ 20 min in class · full notebook ≈ 50 min**

**You will:** (1) package a model as an MLflow **pyfunc** with a *signature*, (2) register versions and manage them with **aliases** (`champion` / `challenger`), (3) serve a registry model with one command and query `/invocations`, (4) promote and **roll back** by moving an alias, (5) score in batch from the CLI, (6) see the container path, (7) build a promotion gate and a **shadow test**.

> Lecture 4a hand-wrote an API. Here MLflow *generates* the API from the model's metadata — you trade flexibility for speed and consistency.

In [ ]:
import sys, subprocess, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlflow>=3", "scikit-learn", "pandas", "joblib", "requests", "matplotlib"], check=False)
os.makedirs("data", exist_ok=True)
os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///mlflow.db"      # the Registry needs a DB-backed store, not plain files
os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"; os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

In [ ]:
%%writefile train_models.py
"""
train_models.py  --  AIOps Module 4 (Model Serving)
Trains TWO versions of a customer-churn classifier on a synthetic, deterministic dataset.
  v1 = LogisticRegression (fast, simple baseline)      -> data/model_v1.joblib
  v2 = RandomForest       (slower, usually more accurate) -> data/model_v2.joblib
Two versions let us practise canary releases, blue-green switches and rollback later.
CPU only; runs in a few seconds.
"""
import json, argparse, numpy as np, pandas as pd, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

FEATURES = ["tenure_months", "monthly_charge", "support_calls", "late_payments",
            "data_usage_gb", "contract_months", "num_products", "age"]

def make_data(n=6000, seed=7):
    r = np.random.default_rng(seed)
    df = pd.DataFrame({
        "tenure_months": r.integers(1, 72, n),
        "monthly_charge": r.normal(65, 20, n).clip(15, 150).round(2),
        "support_calls": r.poisson(2, n),
        "late_payments": r.poisson(0.7, n),
        "data_usage_gb": r.gamma(3, 8, n).round(2),
        "contract_months": r.choice([1, 12, 24], n, p=[.5, .3, .2]),
        "num_products": r.integers(1, 6, n),
        "age": r.integers(18, 80, n),
    })
    z = (-2.0 - 0.02*df.tenure_months + 0.01*df.monthly_charge + 0.15*df.support_calls
         + 0.2*df.late_payments - 0.01*df.contract_months - 0.1*df.num_products
         + 2.2*((df.support_calls >= 4) & (df.contract_months == 1))      # non-linear interactions:
         + 1.8*((df.tenure_months < 12) & (df.monthly_charge > 80))       # a linear model cannot
         + 1.5*((df.late_payments >= 2) & (df.num_products <= 2))         # capture these well
         - 1.6*((df.contract_months == 24) & (df.tenure_months > 36))
         + r.normal(0, .5, n))
    df["churn"] = (1/(1+np.exp(-z)) > r.uniform(size=n)).astype(int)
    return df

def main(out="data"):
    df = make_data()
    X, y = df[FEATURES], df["churn"]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=0, stratify=y)
    models = {
        "v1": make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
        "v2": RandomForestClassifier(n_estimators=150, max_depth=10, random_state=0, n_jobs=1),
    }
    report = {}
    for ver, m in models.items():
        m.fit(Xtr, ytr)
        acc = accuracy_score(yte, m.predict(Xte))
        joblib.dump({"model": m, "feature_columns": FEATURES, "version": ver}, f"{out}/model_{ver}.joblib")
        report[ver] = round(acc, 4)
        print(f"{ver}: test accuracy = {acc:.4f}")
    df.to_csv(f"{out}/churn.csv", index=False)
    Xte.head(200).to_csv(f"{out}/sample_requests.csv", index=False)
    json.dump({"rows": Xte.head(3).to_dict(orient="records")}, open(f"{out}/sample_payload.json", "w"), indent=2)
    json.dump(report, open(f"{out}/metrics.json", "w"))

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--out", default="data"); main(ap.parse_args().out)

In [ ]:
%%writefile mlflow_register.py
"""
mlflow_register.py -- AIOps Module 4, Topic 2: package both churn models as MLflow models,
register them, and assign aliases  champion -> v1,  challenger -> v2.
Uses a local SQLite backend (registry needs a DB-backed store) + local ./mlartifacts.
"""
import os, joblib, mlflow, mlflow.pyfunc, pandas as pd, numpy as np
from mlflow.models import infer_signature
from sklearn.metrics import accuracy_score
from mlflow import MlflowClient

mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db"))
mlflow.set_experiment("churn-serving")
NAME = "churn-model"

class ChurnPyfunc(mlflow.pyfunc.PythonModel):
    """Wraps the joblib bundle so the served model returns probability + label + model version."""
    def load_context(self, context):
        b = joblib.load(context.artifacts["bundle"])
        self.m, self.cols, self.ver = b["model"], b["feature_columns"], b["version"]
    def predict(self, context, model_input, params=None):
        X = pd.DataFrame(model_input)[self.cols]
        p = self.m.predict_proba(X)[:, 1]
        return pd.DataFrame({"churn_probability": p.round(4), "churn": (p >= .5).astype(int),
                             "model_version": self.ver})

df = pd.read_csv("data/churn.csv")
X, y = df.drop(columns="churn"), df["churn"]
Xs = X.astype({c: "float64" for c in X.columns})        # float schema: tolerant to ints/floats from JSON
metrics = {}
client = MlflowClient()
for ver in ("v1", "v2"):
    bundle = f"data/model_{ver}.joblib"
    b = joblib.load(bundle)
    acc = accuracy_score(y, b["model"].predict(X[b["feature_columns"]]))
    out_example = pd.DataFrame({"churn_probability": [0.1], "churn": [0], "model_version": ["v1"]})
    sig = infer_signature(Xs.head(5), out_example)       # explicit input/output schema = serving contract
    with mlflow.start_run(run_name=f"churn-{ver}"):
        mlflow.log_param("algorithm", type(b["model"]).__name__)
        mlflow.log_metric("train_accuracy", acc)
        info = mlflow.pyfunc.log_model(
            name="model", python_model=ChurnPyfunc(), artifacts={"bundle": bundle},
            signature=sig, input_example=Xs.head(2),
            pip_requirements=["scikit-learn", "pandas", "joblib", "numpy"],
            registered_model_name=NAME)
    print(ver, "accuracy", round(acc, 4), "->", info.model_uri)

versions = sorted(client.search_model_versions(f"name='{NAME}'"), key=lambda v: int(v.version))[-2:]
client.set_registered_model_alias(NAME, "champion", versions[0].version)
client.set_registered_model_alias(NAME, "challenger", versions[1].version)
for a in ("champion", "challenger"):
    print(a, "-> version", client.get_model_version_by_alias(NAME, a).version)

In [ ]:
%%writefile load_test.py
"""
load_test.py -- generic closed-loop load generator for any model endpoint (REST / MLflow / Ray / K8s).
  python load_test.py --url http://localhost:8000/v1/models/churn:predict --format rest
  python load_test.py --url http://localhost:5001/invocations --format mlflow --concurrency 1 4 8
Reports p50/p95/p99 latency, throughput, errors and which replicas / model versions answered.
"""
import argparse, json, time, statistics, collections, concurrent.futures as cf
import pandas as pd, requests

def payload(rows, fmt, batch):
    recs = rows[:batch]
    if fmt == "rest":   return {"instances": recs}
    if fmt == "mlflow": return {"dataframe_records": recs}
    raise ValueError(fmt)

def one(sess, url, body):
    t0 = time.perf_counter()
    try:
        r = sess.post(url, json=body, timeout=30)
        ok = r.status_code == 200
        j = r.json() if ok else {}
        who = j.get("served_by", "n/a") if isinstance(j, dict) else "n/a"
        ver = j.get("model_version", "n/a") if isinstance(j, dict) else "n/a"
        if ver == "n/a" and isinstance(j, dict) and j.get("predictions") and isinstance(j["predictions"][0], dict):
            ver = j["predictions"][0].get("model_version", "n/a")      # MLflow pyfunc output
    except Exception:
        ok, who, ver = False, "n/a", "n/a"
    return (time.perf_counter() - t0) * 1000, ok, who, ver

def run(url, fmt, conc, duration, rows, batch):
    body = payload(rows, fmt, batch)
    lat, errs, who, ver = [], 0, collections.Counter(), collections.Counter()
    end = time.time() + duration
    def worker():
        s = requests.Session(); out = []
        while time.time() < end:
            out.append(one(s, url, body))
        return out
    with cf.ThreadPoolExecutor(conc) as ex:
        for res in [f.result() for f in [ex.submit(worker) for _ in range(conc)]]:
            for ms, ok, w, v in res:
                lat.append(ms); errs += (not ok); who[w] += 1; ver[v] += 1
    lat.sort(); n = len(lat); q = lambda p: lat[min(n-1, int(p*n))]
    return dict(concurrency=conc, requests=n, rps=round(n/duration, 1), p50_ms=round(q(.5), 1),
                p95_ms=round(q(.95), 1), p99_ms=round(q(.99), 1), errors=errs,
                replicas=len(who), versions=dict(ver))

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--url", required=True); ap.add_argument("--format", default="rest")
    ap.add_argument("--concurrency", type=int, nargs="+", default=[1, 4, 8])
    ap.add_argument("--duration", type=float, default=5); ap.add_argument("--batch", type=int, default=1)
    ap.add_argument("--data", default="data/sample_requests.csv"); ap.add_argument("--out")
    a = ap.parse_args()
    rows = pd.read_csv(a.data).to_dict(orient="records")
    results = [run(a.url, a.format, c, a.duration, rows, a.batch) for c in a.concurrency]
    print(pd.DataFrame(results).to_string(index=False))
    if a.out: json.dump(results, open(a.out, "w"), indent=2)

In [ ]:
%%writefile m4util.py
"""m4util.py -- tiny helpers shared by the Module 4 notebooks (start / wait for / stop server processes)."""
import subprocess, time, os, signal, sys, requests

_procs = []

def start(cmd, log="server.log", env=None):
    """Start a server as a child process; stdout/stderr go to a log file (read it when things break)."""
    e = {**os.environ, **(env or {})}
    f = open(log, "w")
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=f, stderr=subprocess.STDOUT, env=e,
                         preexec_fn=os.setsid)
    _procs.append(p)
    return p

def wait_ready(url, timeout=90):
    """Poll a URL until it answers 200 (readiness probe, exactly what Kubernetes does)."""
    t0 = time.time()
    while time.time() - t0 < timeout:
        try:
            if requests.get(url, timeout=2).status_code == 200:
                return round(time.time() - t0, 1)
        except Exception:
            pass
        time.sleep(0.5)
    raise TimeoutError(f"{url} not ready after {timeout}s -- check the log file")

def stop(p, timeout=10):
    """Stop one server: SIGTERM the whole process group (so uvicorn/gunicorn workers die too),
    then WAIT until the group is really gone -- otherwise a restart can 'succeed' against the old process."""
    try:
        pgid = os.getpgid(p.pid)
    except Exception:
        return
    try:
        os.killpg(pgid, signal.SIGTERM)
        t0 = time.time()
        while time.time() - t0 < timeout:
            try: os.killpg(pgid, 0)            # signal 0 = "does the group still exist?"
            except ProcessLookupError: break
            time.sleep(0.2)
        else:
            os.killpg(pgid, signal.SIGKILL)
    except ProcessLookupError:
        pass
    try: p.wait(timeout=2)
    except Exception: pass

def stop_all():
    for p in _procs: stop(p)
    _procs.clear()

In [ ]:
!python train_models.py
!python mlflow_register.py 2>&1 | grep -E "^(v|champ|chall)"

## 1 · Inspect what was registered
`mlflow_register.py` did four things per version: wrapped the sklearn bundle in a `PythonModel` (**pyfunc** = MLflow's universal serving interface), inferred a **signature** (the input/output schema), logged it with an `input_example`, and registered it as a new version of `churn-model`. Finally it attached **aliases**.

In [ ]:
import mlflow, pandas as pd, json, requests, time, m4util
from mlflow import MlflowClient
client = MlflowClient(); NAME = "churn-model"
rows = []
for v in client.search_model_versions(f"name='{NAME}'"):
    rows.append(dict(version=v.version, run_id=v.run_id[:8], source=v.source))
alias_map = client.get_registered_model(NAME).aliases            # {'alias': version}
df = pd.DataFrame(rows).sort_values("version")
df["aliases"] = df.version.map(lambda v: [a for a, ver in alias_map.items() if str(ver) == str(v)])
df

In [ ]:
info = mlflow.models.get_model_info(f"models:/{NAME}@champion")
print("signature inputs :", [f"{i.name}:{i.type}" for i in info.signature.inputs][:4], "...")
print("signature outputs:", [f"{o.name}:{o.type}" for o in info.signature.outputs])
print("flavors          :", list(info.flavors))

The signature is the **serving contract**: MLflow enforces it on every request (you will trigger that error shortly). Note the flavor `python_function` — any pyfunc model can be served the same way regardless of the training library.

## 2 · Serve the champion with one command

In [ ]:
CMD = 'mlflow models serve -m "models:/churn-model@champion" -p 5001 --env-manager local'
print(CMD)
srv = m4util.start(CMD, log="mlflow_serve.log")
print("ready after", m4util.wait_ready("http://localhost:5001/ping"), "s")
print(requests.get("http://localhost:5001/version").text)

`--env-manager local` reuses *this* Python environment. In production you would use `virtualenv` / `conda` (MLflow rebuilds the model's pinned environment) or — better — a container (§5).

In [ ]:
sample = pd.read_csv("data/sample_requests.csv")
body = {"dataframe_records": sample.head(3).to_dict(orient="records")}
#print(json.dumps(body, indent=1)[:300])
r = requests.post("http://localhost:5001/invocations", json=body)
print(r.status_code); print(json.dumps(r.json(), indent=1)[:600])

### Schema enforcement — the contract in action

In [ ]:
bad = {
  "missing columns":   {"dataframe_records": [{"tenure_months": 3}]},
  "string in numeric": {"dataframe_records": [{**sample.iloc[0].to_dict(), "monthly_charge": "lots"}]},
  "extra column":      {"dataframe_records": [{**sample.iloc[0].to_dict(), "favourite_colour": "teal"}]},
}
for k, b in bad.items():
    r = requests.post("http://localhost:5001/invocations", json=b)
    print(f"{k:18s} -> HTTP {r.status_code}  {r.json().get('message','ok')[:].replace(chr(10),' ')}")

**Compare with Lecture 4a:** MLflow rejects missing and non-numeric fields (HTTP 400, not 422) but *silently accepts extra columns* and has no range checks (`age=500` is fine). A schema checks **types and presence, not business rules**. If you need range validation or custom routes, wrap the pyfunc in your own FastAPI app — a very common pattern.

## 3 · Promote, then roll back — by moving an alias

In [ ]:
def served_version():
    r = requests.post("http://localhost:5001/invocations", json=body)
    return r.json()["predictions"][0]["model_version"]
print("serving:", served_version())

In [ ]:
# 1. move the alias (metadata only — no files copied, instantaneous)
champ_before = client.get_model_version_by_alias(NAME, "champion").version
chall = client.get_model_version_by_alias(NAME, "challenger").version
client.set_registered_model_alias(NAME, "champion", chall)
print("champion now points to version", client.get_model_version_by_alias(NAME, "champion").version)
print("...but the running server still answers with:", served_version(), " <- alias is resolved at START-UP")

In [ ]:
# 2. restart the server so it re-resolves the alias
m4util.stop(srv)
srv = m4util.start(CMD, log="mlflow_serve.log"); m4util.wait_ready("http://localhost:5001/ping")
print("after restart  :", served_version())

In [ ]:
# 3. ROLLBACK = move the alias back and restart. Same two steps, same speed.
client.set_registered_model_alias(NAME, "champion", champ_before)
m4util.stop(srv)
srv = m4util.start(CMD, log="mlflow_serve.log"); m4util.wait_ready("http://localhost:5001/ping")
print("after rollback :", served_version())

**Takeaway.** Aliases decouple *what clients ask for* (`champion`) from *which build answers*. Rollback needs no rebuild — but note the restart: a bare `mlflow models serve` process does not hot-reload. In Kubernetes (Lecture 4d) a rolling update does that restart *without downtime*.

## 4 · Offline batch scoring from the CLI

In [ ]:
json.dump(body, open("batch_in.json", "w"))
!mlflow models predict -m "models:/churn-model@champion" -i batch_in.json -t json --env-manager local -o batch_out.json 2>&1 | tail -2
print(open("batch_out.json").read()[:300])

Same model artifact, no server: the right tool for nightly scoring jobs. **Online (REST) vs batch** is a deployment decision about latency needs, not a model difference.

## 5 · The container path (read; run if Docker is available)

In [ ]:
import shutil
DOCKER_CMD = 'mlflow models build-docker -m "models:/churn-model@champion" -n churn-mlflow:champion --env-manager virtualenv'
print(DOCKER_CMD)
print("docker daemon reachable:", subprocess.run("docker info", shell=True, capture_output=True).returncode == 0)
# then:  docker run -p 5001:8080 churn-mlflow:champion     (the container serves on 8080)

The resulting image contains the model, its pinned environment and the same `/invocations` server — the exact artifact you would load into minikube in Lecture 4d (`minikube image load churn-mlflow:champion`).

## 6 · Load test MLflow serving

In [ ]:
import load_test
rows_all = sample.to_dict(orient="records")
res = [load_test.run("http://localhost:5001/invocations", "mlflow", cc, 4, rows_all, 1) for cc in [1, 4, 8]]
pd.DataFrame(res)[["concurrency", "requests", "rps", "p50_ms", "p95_ms", "p99_ms", "errors", "versions"]]

Compare with Lecture 4a's FastAPI numbers: MLflow's pyfunc server adds a DataFrame conversion and schema enforcement per request, so single-row latency is typically a bit higher. Convenience has a cost — measure it for *your* model rather than assuming.

## 7 · YOUR TURN

**Task A — a promotion gate.** Write `promotion_gate(min_gain=0.01)` that evaluates *champion* and *challenger* on **fresh held-out data** (`train_models.make_data(seed=99)`), and returns a dict `{champion_acc, challenger_acc, promote: bool}`. Promote only if the challenger improves accuracy by at least `min_gain`. (Why must the data be fresh? Look at the `train_accuracy` MLflow logged for v2.)

**Task B — a shadow test.** Start the *challenger* on port 5002. Send the same 200 requests to both servers and report (i) the **disagreement rate** between predicted labels, and (ii) each server's p95 latency. Clients only ever see the champion's answer — the challenger is observed, not trusted.

In [ ]:
# Task A — your code here

**Solution A**

In [ ]:
import train_models
from sklearn.metrics import accuracy_score
fresh = train_models.make_data(seed=99)
def promotion_gate(min_gain=0.01):
    accs = {}
    for alias in ("champion", "challenger"):
        m = mlflow.pyfunc.load_model(f"models:/{NAME}@{alias}")
        pred = m.predict(fresh[train_models.FEATURES].astype("float64"))["churn"]
        accs[alias] = accuracy_score(fresh["churn"], pred)
    return dict(champion_acc=round(accs["champion"], 4), challenger_acc=round(accs["challenger"], 4),
                promote=bool(accs["challenger"] - accs["champion"] >= min_gain))
print(promotion_gate(0.01)); print(promotion_gate(0.10))

In [ ]:
# Task B — your code here

**Solution B**

In [ ]:
srv_b = m4util.start('mlflow models serve -m "models:/churn-model@challenger" -p 5002 --env-manager local', log="mlflow_serve_b.log")
m4util.wait_ready("http://localhost:5002/ping")
import numpy as np
lat = {5001: [], 5002: []}; labels = {5001: [], 5002: []}
for rec in rows_all[:200]:
    for port in (5001, 5002):
        t = time.perf_counter()
        r = requests.post(f"http://localhost:{port}/invocations", json={"dataframe_records": [rec]})
        lat[port].append((time.perf_counter() - t) * 1000); labels[port].append(r.json()["predictions"][0]["churn"])
dis = np.mean(np.array(labels[5001]) != np.array(labels[5002]))
print(f"disagreement rate: {dis:.1%}")
print(f"p95 latency  champion: {np.percentile(lat[5001], 95):.1f} ms   challenger: {np.percentile(lat[5002], 95):.1f} ms")

A non-zero disagreement rate is expected (and informative). Paired with delayed ground truth, a shadow run tells you whether to promote **before** any user is affected.

In [ ]:
m4util.stop_all(); print("stopped")

### Recap
* pyfunc + signature = portable, enforceable serving contract (types, not business rules).
* Registry **aliases** make promotion and rollback metadata operations; the server resolves them at start-up.
* `mlflow models serve` for dev/test; `build-docker` for deployment; `mlflow models predict` for batch.
* Gate promotions on **fresh** data; use **shadow** traffic to compare safely.

**Next:** Lecture 4c — Ray Serve: replicas, autoscaling and traffic splitting inside one Python program.